# DBRepo exercise notebook: ingest data and run example queries

This notebook is designed as a **step-by-step guide** for students who need to ingest data into **DBRepo** and then query it with the **DBRepo Python API**.

It follows the DBRepo 1.13 documentation and uses the Python client methods shown there, especially:

- `RestClient(...)`
- `create_table(...)`
- `get_tables(...)`
- `get_table_data(...)`
- `import_table_data(...)`
- `create_view(...)`
- `get_view_data(...)`
- `create_subset(...)`
- `get_subset_data(...)`



## What students will learn

Students will get an overview over the main functions of the DBRepo Python Library. Note that this guide is only a showcase and has nothing to do with the exercise itself.

This is the correct version of the library: https://pypi.org/manage/project/dbrepo/release/1.13.5/


## Cell 1 — Install the Python package

Run this only if `dbrepo` is not installed yet.  
Adjust the version to the one your DBRepo instance expects.

According to the DBRepo docs, the Python library version should match the instance version shown in the UI.


In [ ]:
# Uncomment and run if needed:
# %pip install dbrepo==1.13.5 pandas

# If your instance uses a slightly different 1.13.x version,
# install that exact version instead.


## Cell 2 — Imports

We import the DBRepo client, the DTOs needed for query definitions, and pandas for local data preparation.


In [3]:
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import (
    QueryDefinition,
    FilterDefinition,
    FilterType,
    OrderDefinition,
    OrderType,
)
import pandas as pd
import time



## Cell 3 — Connection settings

Fill in your DBRepo endpoint and credentials.

Examples:
- hosted test instance: `https://test.dbrepo.tuwien.ac.at`
- local/self-hosted instance: something like `http://localhost`

On the test instance click login and register in the UI. Note, that after registration the redirect in the browser does not work. From there, we change to the API. Insert your username and password in the cell below.

Furthermore, you need to create your own database manually. Therefore use the endpoints from the Swagger UI. Use the same credentials there. Make sure to choose private to not allow others to see your data! The DATABASE_ID will be in the response and needs to be pasted here.


In [ ]:
ENDPOINT = "https://test.dbrepo.tuwien.ac.at"   # <-- change if needed
USERNAME = "YOUR_USERNAME"                            # <-- change
PASSWORD = "YOUR_PASSWORD"                           # <-- change
DATABASE_ID = "DATABASE_ID"                      # <-- change

client = RestClient(endpoint=ENDPOINT, username=USERNAME, password=PASSWORD)


## Cell 4 — Check the connection

This is a good first sanity check. If it works, the client can authenticate and talk to DBRepo.


In [5]:
print("Current user:", client.whoami())


mhoeller
Current user: mhoeller


## Cell 5 — Optional: inspect visible databases

This helps students understand what databases they can currently see.


In [6]:
databases = client.get_databases()
print(f"Visible databases: {len(databases)}")

for db in databases[:10]:
    print(f"- {db.id} | {db.name}")


Visible databases: 4
- aeadd4f4-ddcd-4ca7-9264-4c095f753799 | demo_db
- 7d7a1d82-62b5-48b3-9e75-6940fb9d0e30 | students_demo_database
- 81d82941-cae0-4cba-b27d-1bd883dd713a | data_stew_grp22_air_quality
- 17de844a-62fc-406f-abb2-fffb22b44d01 | meine-datenbank


## Cell 6 — Prepare a tiny example dataset locally

For teaching, it is useful to start with a very small DataFrame that students can understand immediately.

You can replace this with the real exercise dataset later.


In [7]:
df_local = pd.DataFrame(
    [
        {"student_id": 1, "name": "Anna",  "course": "DS101", "score": 84},
        {"student_id": 2, "name": "Ben",   "course": "DS101", "score": 91},
        {"student_id": 3, "name": "Clara", "course": "DS102", "score": 78},
        {"student_id": 4, "name": "David", "course": "DS102", "score": 88},
        {"student_id": 5, "name": "Elias", "course": "DS101", "score": 95},
    ]
)
df_local = df_local.set_index('student_id')

df_local


,name,course,score
student_id,,,
1,Anna,DS101,84
2,Ben,DS101,91
3,Clara,DS102,78
4,David,DS102,88
5,Elias,DS101,95


## Cell 7 — Optional: inspect the local schema

Before ingestion, it is useful to look at data types and column names.


In [8]:
print(df_local.dtypes)
print("\nColumns:", list(df_local.columns))


name        str
course      str
score     int64
dtype: object

Columns: ['name', 'course', 'score']


## Cell 8 — Create a table in DBRepo from the DataFrame

This is the core ingestion step.

`create_table(...)` creates a DBRepo table from the pandas DataFrame.  
We keep both `is_public` and `is_schema_public` set to `False` for a classroom exercise by default.


In [14]:
table = client.create_table(
    database_id=DATABASE_ID,
    name="student_demo",
    is_public=False,
    is_schema_public=False,
    dataframe=df_local,
    description="Small demo table for DBRepo ingestion exercise",
)

print("Created table id:", table.id)
print("Created table name:", table.name)
TABLE_ID = table.id
TABLE_NAME = table.name


2026-04-27 10:43:37,251 root         WARNING default to 'text' for column name and type <class 'numpy.dtype'>
2026-04-27 10:43:37,253 root         WARNING default to 'text' for column course and type <class 'numpy.dtype'>
Created table id: 930b2c1f-edc1-4d72-a44e-7204ce81e970
Created table name: student_demo


## Cell 9 — List tables in the selected database

Students can verify that the new table now exists inside the DBRepo database.


In [15]:
tables = client.get_tables(DATABASE_ID)
print(f"Tables in database: {len(tables)}")

for t in tables:
    print(f"- {t.id} | {t.name}")


Tables in database: 1
- 930b2c1f-edc1-4d72-a44e-7204ce81e970 | student_demo


## Cell 10 — Read the data back from DBRepo

A very important habit after ingestion: immediately fetch the table back and verify the result.


In [16]:
print("Waiting for asynchronous data import to finish...")
time.sleep(5)  # Wait for DBRepo queue to process the S3 file

# Optional polling:
for _ in range(10):
    try:
        df_from_dbrepo = client.get_table_data(
            database_id=DATABASE_ID,
            table_id=TABLE_ID,
            page=0,
            size=1000000,
        )
        break
    except Exception as e:
        if '204' in str(e):
            print("Still waiting...")
            time.sleep(5)
        else:
            raise e


df_from_dbrepo


Waiting for asynchronous data import to finish...


,student_id,name,course,score
0,1,Anna,DS101,84
1,2,Ben,DS101,91
2,3,Clara,DS102,78
3,4,David,DS102,88
4,5,Elias,DS101,95


## Cell 11 — Check row count

This shows how many rows DBRepo currently reports for the table.


In [17]:
row_count = client.get_table_data_count(
    database_id=DATABASE_ID,
    table_id=TABLE_ID,
)

print("Row count in DBRepo table:", row_count)


Row count in DBRepo table: 5


## Cell 12 — Import additional rows into the same table

The DBRepo docs warn that `import_table_data(...)` is **column-order sensitive**.  
So we construct the new dataframe with the **same columns in the same order** as the existing table.


In [19]:
df_more = pd.DataFrame(
    [
        {"student_id": 6, "name": "Fatima", "course": "DS103", "score": 82},
        {"student_id": 7, "name": "Gregor", "course": "DS101", "score": 89},
    ]
)

df_more = df_more[["student_id", "name", "course", "score"]]

client.import_table_data(
    database_id=DATABASE_ID,
    table_id=TABLE_ID,
    dataframe=df_more,
)

print("Additional rows imported.")


Additional rows imported.


## Cell 13 — Read the table again after the append

Now the table should contain the original rows plus the new ones.


In [20]:
print("Waiting for asynchronous append to finish...")
time.sleep(5)

for _ in range(10):
    try:
        df_after_append = client.get_table_data(
            database_id=DATABASE_ID,
            table_id=TABLE_ID,
            page=0,
            size=1000000,
        )
        break
    except Exception as e:
        if '204' in str(e):
            print("Still waiting for append...")
            time.sleep(5)
        else:
            raise e

df_after_append


Waiting for asynchronous append to finish...


,student_id,name,course,score
0,1,Anna,DS101,84
1,2,Ben,DS101,91
2,3,Clara,DS102,78
3,4,David,DS102,88
4,5,Elias,DS101,95
5,6,Fatima,DS103,82
6,7,Gregor,DS101,89


## Cell 14 — Create a simple view

A **view** stores a reusable query.  
Here we create a view that keeps only students with `score >= 85` and sorts them by score descending.

We use a `QueryDefinition` with:
- one datasource (`students_demo`)
- selected columns
- a `WHERE` filter
- an `ORDER BY`


In [24]:
view_query = QueryDefinition(
    datasources=[TABLE_NAME],
    columns=[
        f"{TABLE_NAME}.student_id",
        f"{TABLE_NAME}.name",
        f"{TABLE_NAME}.course",
        f"{TABLE_NAME}.score",
    ],
    filters=[
        FilterDefinition(
            type=FilterType.WHERE,
            column=f"{TABLE_NAME}.score",
            operator=">=",
            value="85",
        )
    ],
    orders=[
        OrderDefinition(
            column=f"{TABLE_NAME}.score",
            direction=OrderType.DESC,
        )
    ],
)

view = client.create_view(
    database_id=DATABASE_ID,
    name="high_scores_view",
    query=view_query,
    is_public=False,
    is_schema_public=False,
)

print("Created view id:", view.id)
print("Created view name:", view.name)

VIEW_ID = view.id


Created view id: 3a6866be-44cd-49a3-b8c6-f8fd38180f56
Created view name: high_scores_view


## Cell 15 — Fetch data from the view

This returns the view result directly as a pandas DataFrame.


In [25]:
print("Waiting for asynchronous view creation to map...")
time.sleep(5)

for _ in range(10):
    try:
        df_view = client.get_view_data(
            database_id=DATABASE_ID,
            view_id=VIEW_ID,
            page=0,
            size=1000000,
        )
        break
    except Exception as e:
        if '204' in str(e):
            print("Still waiting for view...")
            time.sleep(5)
        else:
            raise e

df_view


Waiting for asynchronous view creation to map...


,course,student_id,score,name
0,DS101,2,91,Ben
1,DS102,4,88,David
2,DS101,5,95,Elias
3,DS101,7,89,Gregor


## Cell 16 — Create a subset query

A **subset** is a query result for part of a dataset.  
For the classroom exercise, we create a subset that keeps only course `DS101`.

Unlike a view, `create_subset(...)` directly executes the query and returns a DataFrame result.


In [27]:
subset_query = QueryDefinition(
    datasources=[TABLE_NAME],
    columns=[
        f"{TABLE_NAME}.student_id",
        f"{TABLE_NAME}.name",
        f"{TABLE_NAME}.course",
        f"{TABLE_NAME}.score",
    ],
    filters=[
        FilterDefinition(
            type=FilterType.WHERE,
            column=f"{TABLE_NAME}.course",
            operator="=",
            value="DS101",
        )
    ],
    orders=[
        OrderDefinition(
            column=f"{TABLE_NAME}.student_id",
            direction=OrderType.ASC,
        )
    ],
)

df_subset = client.create_subset(
    database_id=DATABASE_ID,
    query=subset_query,
    page=0,
    size=1000000,
)

df_subset


2026-04-27 10:48:20,431 root         INFO   Created subset with id: d6fb515a-4215-11f1-ad46-b69aff86b1da


,course,student_id,score,name
0,DS101,1,84,Anna
1,DS101,2,91,Ben
2,DS101,5,95,Elias
3,DS101,7,89,Gregor


## Cell 17 — Optional: inspect metadata for the table and the view

Students often benefit from seeing that DBRepo stores metadata objects, not just raw rows.


In [32]:
table_meta = client.get_table(DATABASE_ID, TABLE_ID)
view_meta = client.get_view(DATABASE_ID, VIEW_ID)

table_meta, view_meta


(Table(id='930b2c1f-edc1-4d72-a44e-7204ce81e970', database_id='aeadd4f4-ddcd-4ca7-9264-4c095f753799', name='student_demo', owner=UserBrief(username='mhoeller', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None), columns=[Column(id='2ddef186-7c9d-4df9-b282-94e4cf7a4464', name='student_id', database_id='aeadd4f4-ddcd-4ca7-9264-4c095f753799', table_id='930b2c1f-edc1-4d72-a44e-7204ce81e970', ord=0, internal_name='student_id', is_null_allowed=False, type=<ColumnType.BIGINT: 'bigint'>, alias=None, description=None, size=None, d=None, mean=4.0, median=4.0, concept=None, unit=None, enums=[], sets=[], index_length=None, length=None, data_length=None, max_data_length=None, num_rows=None, val_min=None, val_max=None, std_dev=2.0), Column(id='636c92e3-2ebe-43fb-abb4-953cc1995cb4', name='name', database_id='aeadd4f4-ddcd-4ca7-9264-4c095f753799', table_id='930b2c1f-edc1-4d72-a44e-7204ce81e970', ord=1, internal_name='name', is_null_allowed=False, type=<ColumnType.T

## Cell 18 — Optional cleanup

Only run this if you want to remove the exercise objects afterwards.


In [33]:
client.delete_view(DATABASE_ID, VIEW_ID)
client.delete_table(DATABASE_ID, TABLE_ID)
print("Cleanup done.")


Cleanup done.
